# IAN 601 — Module 12: Cleaning & Exploratory Data Analysis

Real-world data is often incomplete, inconsistent, duplicated, incorrectly typed, or simply surprising.

Data cleaning is the process of finding and addressing those issues deliberately. Exploratory data analysis, or **EDA**, is the process of examining the cleaned data to understand its distributions, categories, unusual values, and possible relationships.

This notebook is self-contained and uses one intentionally messy dataset so the full process can be practiced from audit to validation.

## Learning goals

By the end of this notebook, you should be able to:

1. perform a basic data-quality audit;
2. measure missingness;
3. identify duplicate records;
4. convert numeric and date columns safely;
5. standardize inconsistent text categories;
6. make deliberate missing-value decisions;
7. validate simple ranges and business rules;
8. identify possible outliers with the IQR rule;
9. explore categorical and numeric variables;
10. compare groups;
11. build a compact cleaning pipeline;
12. validate the dataset after cleaning.

## How to use this notebook

Run the cells from top to bottom the first time so that variables and imported libraries are available when later cells need them.

As you work:

1. read the explanation before the code;
2. predict the result;
3. run the cell;
4. inspect the output carefully;
5. change one value, condition, or option;
6. rerun the cell and explain what changed.

For notebooks that load a CSV from GitHub, the dataset is loaded automatically. If you restart the runtime, run the notebook again from the beginning.

### Expandable syntax guides

Each major section includes a collapsed **Syntax & parameters** box. Open it for function signatures, important parameters, related methods, and practical cautions. Keep it closed when you want to focus on the main cleaning workflow.

## 1. Load the messy dataset

<details>
<summary><strong>Syntax & parameters — loading messy CSV data</strong></summary>

<pre><code class="language-python">pd.read_csv(
    "file.csv",
    na_values=None,
    dtype=None,
    parse_dates=None,
    keep_default_na=True
)</code></pre>

<p><strong>Useful parameters for messy files</strong></p>
<ul>
<li><code>na_values</code> — strings such as <code>"N/A"</code>, <code>"missing"</code>, or <code>"?"</code> to treat as missing.</li>
<li><code>dtype</code> — force a column to load as a specific type.</li>
<li><code>parse_dates</code> — parse date columns during import.</li>
<li><code>usecols</code> — load selected columns only.</li>
<li><code>skiprows</code> — skip metadata rows above a table.</li>
<li><code>encoding</code> — specify text encoding.</li>
</ul>

<p>For an audit, it is often useful to load first and convert types explicitly afterward so you can see the original problems.</p>
</details>

In [ ]:
import pandas as pd
import numpy as np

data_url = "https://raw.githubusercontent.com/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/main/IAN601_Module12_Cleaning_EDA_400.csv"
orders = pd.read_csv(data_url)

orders.head()

## 2. Start with a data audit

Do not fix anything yet. First understand what is wrong.

<details>
<summary><strong>Syntax & parameters — data audit toolkit</strong></summary>

<pre><code class="language-python">df.shape
df.head()
df.sample(5, random_state=1)
df.columns
df.dtypes
df.info()
df.describe()
df.describe(include="all")</code></pre>

<p><strong>Additional audit helpers</strong></p>
<pre><code class="language-python">df.nunique()
df.isna().sum()
df.duplicated().sum()
df["category"].value_counts(dropna=False)</code></pre>

<p>An audit should answer: What is the unit of one row? Which columns are keys? Which columns are numeric, categorical, or dates? What problems are visible before cleaning?</p>
</details>

In [ ]:
print("Shape:", orders.shape)
print("\nColumns:", orders.columns.tolist())
print("\nData types:")
print(orders.dtypes)

In [ ]:
orders.head(10)

## 3. Missing values

<details>
<summary><strong>Syntax & parameters — missing-value profiling</strong></summary>

<pre><code class="language-python">df.isna()
df.isna().sum()
df.isna().mean() * 100
df.notna().sum()</code></pre>

<p><strong>By row</strong></p>
<pre><code class="language-python">df.isna().sum(axis=1)</code></pre>

<p><strong>Filter rows with missing values</strong></p>
<pre><code class="language-python">df[df["amount"].isna()]
df[df.isna().any(axis=1)]</code></pre>

<p>Missingness can be informative. Before filling or dropping values, ask whether the missingness itself might represent a business process or data-collection issue.</p>
</details>

In [ ]:
missing = pd.DataFrame({
    "missing_count": orders.isna().sum(),
    "missing_percent": (orders.isna().mean() * 100).round(1)
})

missing

## 4. Duplicate records

<details>
<summary><strong>Syntax & parameters — duplicated() and drop_duplicates()</strong></summary>

<pre><code class="language-python">df.duplicated()
df.duplicated(subset=["order_id"], keep=False)

df.drop_duplicates()
df.drop_duplicates(
    subset=["order_id"],
    keep="first"
)</code></pre>

<p><strong>keep options</strong></p>
<ul>
<li><code>"first"</code> — keep the first occurrence.</li>
<li><code>"last"</code> — keep the last occurrence.</li>
<li><code>False</code> — mark or remove every member of a duplicate group.</li>
</ul>

<p>Exact duplicate rows and duplicate business keys are different problems. A repeated <code>order_id</code> may be important even when other columns differ.</p>
</details>

In [ ]:
print("Exact duplicate rows:", orders.duplicated().sum())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

In [ ]:
orders[orders.duplicated(keep=False)].sort_values("order_id")

## 5. Convert data types safely

<details>
<summary><strong>Syntax & parameters — safe type conversion</strong></summary>

<p><strong>Numeric conversion</strong></p>
<pre><code class="language-python">pd.to_numeric(
    df["amount"],
    errors="coerce"
)</code></pre>

<p><strong>Date conversion</strong></p>
<pre><code class="language-python">pd.to_datetime(
    df["order_date"],
    errors="coerce",
    format=None
)</code></pre>

<p><strong>General conversion</strong></p>
<pre><code class="language-python">df["status"] = df["status"].astype("string")
df = df.convert_dtypes()</code></pre>

<p><code>errors="coerce"</code> turns invalid values into missing values rather than crashing. This is useful during cleaning because invalid records become easy to identify.</p>

<p>After conversion, always check how many new missing values were introduced.</p>
</details>

In [ ]:
clean = orders.copy()

clean["amount"] = pd.to_numeric(
    clean["amount"].astype(str).str.replace("$", "", regex=False),
    errors="coerce"
)

clean["order_date"] = pd.to_datetime(
    clean["order_date"],
    errors="coerce"
)

clean["satisfaction"] = pd.to_numeric(
    clean["satisfaction"],
    errors="coerce"
)

clean.dtypes

## 6. Clean text categories

<details>
<summary><strong>Syntax & parameters — Pandas string cleaning</strong></summary>

<pre><code class="language-python">df["status"].str.strip()
df["status"].str.lower()
df["region"].str.title()
df["text"].str.replace("old", "new", regex=False)</code></pre>

<p><strong>Other useful .str methods</strong></p>
<ul>
<li><code>str.upper()</code></li>
<li><code>str.contains()</code></li>
<li><code>str.startswith()</code></li>
<li><code>str.endswith()</code></li>
<li><code>str.split()</code></li>
<li><code>str.extract()</code></li>
<li><code>str.len()</code></li>
</ul>

<p>Use <code>value_counts(dropna=False)</code> before and after standardization to verify that category variants were actually consolidated.</p>
</details>

In [ ]:
clean["region"] = clean["region"].str.strip().str.title()
clean["channel"] = clean["channel"].str.strip().str.lower()
clean["status"] = clean["status"].str.strip().str.lower()

print(clean["region"].value_counts(dropna=False))
print()
print(clean["channel"].value_counts(dropna=False))
print()
print(clean["status"].value_counts(dropna=False))

## 7. Handle missing values deliberately

Here we use simple teaching decisions. In real projects, the decision should depend on context and business meaning.

<details>
<summary><strong>Syntax & parameters — fillna(), dropna(), and interpolation</strong></summary>

<p><strong>Fill missing values</strong></p>
<pre><code class="language-python">df["city"] = df["city"].fillna("Unknown")
df["amount"] = df["amount"].fillna(df["amount"].median())</code></pre>

<p><strong>Drop missing values</strong></p>
<pre><code class="language-python">df.dropna()
df.dropna(subset=["order_id"])
df.dropna(thresh=5)</code></pre>

<p><strong>Interpolate ordered numeric data</strong></p>
<pre><code class="language-python">df["value"].interpolate()</code></pre>

<p>Do not choose a method only because it removes <code>NaN</code>. The correct action depends on what the field means and why it is missing.</p>
</details>

In [ ]:
clean["city"] = clean["city"].fillna("Unknown")
clean["region"] = clean["region"].fillna("Unknown")
clean["amount"] = clean["amount"].fillna(clean["amount"].median())

clean.isna().sum()

## 8. Validate ranges and business rules

A satisfaction score should be between 1 and 5.

<details>
<summary><strong>Syntax & parameters — rule validation</strong></summary>

<pre><code class="language-python">df["score"].between(1, 5)
df["status"].isin(["complete", "cancelled"])
df["amount"].ge(0)
df["quantity"].gt(0)</code></pre>

<p><strong>Find invalid rows</strong></p>
<pre><code class="language-python">invalid = df[~df["score"].between(1, 5)]</code></pre>

<p><strong>Assertions</strong></p>
<pre><code class="language-python">assert df["order_id"].notna().all()
assert df["score"].dropna().between(1, 5).all()</code></pre>

<p>Other useful tools include <code>clip()</code>, <code>where()</code>, and <code>mask()</code>, but do not silently force invalid values into a valid range unless that rule is justified.</p>
</details>

In [ ]:
invalid_satisfaction = clean[
    clean["satisfaction"].notna()
    & ~clean["satisfaction"].between(1, 5)
]

invalid_satisfaction[["order_id", "satisfaction"]]

In [ ]:
clean.loc[
    ~clean["satisfaction"].between(1, 5),
    "satisfaction"
] = np.nan

clean["satisfaction"].describe()

## 9. Outliers are questions, not automatic deletions

<details>
<summary><strong>Syntax & parameters — IQR outlier rule</strong></summary>

<pre><code class="language-python">q1 = s.quantile(0.25)
q3 = s.quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

mask = (s &lt; lower) | (s &gt; upper)</code></pre>

<p>The factor <code>1.5</code> is a common convention, not a universal law.</p>

<p><strong>Other approaches</strong></p>
<ul>
<li>domain-based limits;</li>
<li>z-scores for approximately normal data;</li>
<li>percentile rules;</li>
<li>visual inspection with boxplots or scatter plots.</li>
</ul>

<p>An outlier may be an error, a rare valid case, or the most interesting observation in the dataset.</p>
</details>

In [ ]:
clean["amount"].describe()

### Flag outliers with the IQR rule

In [ ]:
q1 = clean["amount"].quantile(0.25)
q3 = clean["amount"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = clean[
    (clean["amount"] < lower)
    | (clean["amount"] > upper)
]

print("Lower bound:", round(lower, 2))
print("Upper bound:", round(upper, 2))
print("Flagged rows:", len(outliers))

outliers.head()

## 10. Explore categorical variables

<details>
<summary><strong>Syntax & parameters — categorical EDA</strong></summary>

<pre><code class="language-python">df["channel"].value_counts()
df["channel"].value_counts(
    normalize=True,
    dropna=False
)</code></pre>

<p><strong>Two-way counts</strong></p>
<pre><code class="language-python">pd.crosstab(
    df["region"],
    df["channel"],
    margins=True
)</code></pre>

<p><strong>Useful questions</strong></p>
<ul>
<li>Which categories are most common?</li>
<li>Are labels unexpectedly duplicated because of spelling or case?</li>
<li>Are some categories extremely rare?</li>
<li>Does category frequency differ across another variable?</li>
</ul>
</details>

In [ ]:
clean["channel"].value_counts()

In [ ]:
clean["region"].value_counts()

## 11. Explore numeric variables

<details>
<summary><strong>Syntax & parameters — numeric EDA</strong></summary>

<pre><code class="language-python">df["amount"].describe()
df["amount"].quantile([0.25, 0.5, 0.75])
df["amount"].mean()
df["amount"].median()
df["amount"].std()</code></pre>

<p><strong>Quick plots</strong></p>
<pre><code class="language-python">df["amount"].plot(kind="hist", bins=20)
df.boxplot(column="amount")</code></pre>

<p>Center, spread, shape, and unusual values should be considered together. A single statistic rarely tells the full story.</p>
</details>

In [ ]:
clean[["amount", "satisfaction"]].describe()

## 12. Compare groups during EDA

<details>
<summary><strong>Syntax & parameters — group comparisons</strong></summary>

<pre><code class="language-python">df.groupby("channel", as_index=False).agg(
    orders=("order_id", "count"),
    avg_amount=("amount", "mean"),
    median_amount=("amount", "median")
)</code></pre>

<p><strong>Useful comparison statistics</strong></p>
<ul>
<li>count</li>
<li>mean</li>
<li>median</li>
<li>standard deviation</li>
<li>minimum/maximum</li>
<li>unique count</li>
</ul>

<p>Group comparisons can reveal structure that disappears in an overall average. Always consider group sizes when interpreting differences.</p>
</details>

In [ ]:
channel_summary = (
    clean.groupby("channel", as_index=False)
    .agg(
        orders=("order_id", "count"),
        avg_amount=("amount", "mean"),
        avg_satisfaction=("satisfaction", "mean")
    )
)

channel_summary.round(2)

In [ ]:
region_summary = (
    clean.groupby("region", as_index=False)
    .agg(
        orders=("order_id", "count"),
        total_amount=("amount", "sum")
    )
    .sort_values("total_amount", ascending=False)
)

region_summary.round(2)

## 13. Mini cleaning pipeline

A compact pipeline makes the sequence easy to review and reuse.

<details>
<summary><strong>Syntax guide — organizing a cleaning pipeline</strong></summary>

<p>A cleaning pipeline often follows this order:</p>

<pre><code class="language-python">clean = (
    raw
    .copy()
    # standardize types
    # standardize text
    # handle duplicates
    # handle missing values
    # validate rules
)</code></pre>

<p>Long pipelines can be broken into functions and connected with <code>pipe()</code>:</p>

<pre><code class="language-python">clean = (
    raw
    .pipe(clean_types)
    .pipe(clean_labels)
    .pipe(validate_ranges)
)</code></pre>

<p>During development, inspect intermediate results. A pipeline should be reproducible, but it should also be understandable.</p>
</details>

In [ ]:
pipeline = orders.copy()

pipeline["amount"] = pd.to_numeric(
    pipeline["amount"].astype(str).str.replace("$", "", regex=False),
    errors="coerce"
)
pipeline["order_date"] = pd.to_datetime(
    pipeline["order_date"],
    errors="coerce"
)
pipeline["satisfaction"] = pd.to_numeric(
    pipeline["satisfaction"],
    errors="coerce"
)

pipeline["region"] = pipeline["region"].str.strip().str.title()
pipeline["channel"] = pipeline["channel"].str.strip().str.lower()
pipeline["status"] = pipeline["status"].str.strip().str.lower()

pipeline = pipeline.drop_duplicates().copy()

pipeline["city"] = pipeline["city"].fillna("Unknown")
pipeline["region"] = pipeline["region"].fillna("Unknown")
pipeline["amount"] = pipeline["amount"].fillna(
    pipeline["amount"].median()
)

pipeline.loc[
    ~pipeline["satisfaction"].between(1, 5),
    "satisfaction"
] = np.nan

pipeline.head()

## 14. Check the data after cleaning

<details>
<summary><strong>Syntax & parameters — post-cleaning validation</strong></summary>

<pre><code class="language-python">len(clean)
clean.shape
clean.isna().sum()
clean.duplicated().sum()
clean["order_id"].nunique()
clean["amount"].describe()</code></pre>

<p><strong>Compare before and after</strong></p>
<pre><code class="language-python">print("Rows before:", len(raw))
print("Rows after:", len(clean))</code></pre>

<p>Useful validation questions:</p>
<ul>
<li>Did row count change? Why?</li>
<li>Did missingness decrease in the intended fields?</li>
<li>Were invalid categories removed or standardized?</li>
<li>Did totals change unexpectedly?</li>
<li>Are key columns still unique when they should be?</li>
</ul>
</details>

In [ ]:
print("Rows:", len(pipeline))
print("Duplicate rows:", pipeline.duplicated().sum())
print("Missing amount:", pipeline["amount"].isna().sum())
print("Invalid dates:", pipeline["order_date"].isna().sum())
print("Invalid satisfaction values:",
      (~pipeline["satisfaction"].dropna().between(1, 5)).sum())

# Optional Study — Polars for Data Cleaning & EDA

**Polars** is another DataFrame library for Python.

Like Pandas, it can read files, select and filter columns, clean strings, convert data types, handle missing values, remove duplicates, group and aggregate data, join tables, and reshape data.

Polars is especially interesting for larger analytical workloads because it is written in Rust, is designed to use multiple CPU cores for many operations, and includes a query optimizer for lazy workflows.

That does **not** mean Polars is always faster.

For a small dataset such as the 400-row teaching file in this notebook, performance differences are usually not important. Polars becomes more interesting when data and workflows become larger, repeated, or computationally expensive.

The goal here is not to replace Pandas. It is to learn that the Python DataFrame ecosystem has more than one tool.

## Pandas and Polars: a quick comparison

| Topic | Pandas | Polars |
|---|---|---|
| Main object | `DataFrame` | `DataFrame` / `LazyFrame` |
| Common column syntax | `df["amount"]` | `pl.col("amount")` |
| Filter | Boolean indexing / `query()` | `filter()` |
| Create columns | assignment / `assign()` | `with_columns()` |
| Grouping | `groupby()` | `group_by()` |
| Missing marker | often `NaN` / `pd.NA` | `null` |
| Lazy query planning | not the main API model | major part of the API |
| Multi-core analytical execution | varies by operation | designed for parallel execution across many operations |
| Ecosystem | very broad and widely taught | newer and focused on high-performance DataFrame work |

### When might Polars be attractive?

Consider Polars when the dataset is large, the workflow has many filters or aggregations, memory use matters, repeated DataFrame operations are becoming slow, or a lazy query plan can eliminate unnecessary work.

### When might Pandas still be preferable?

Pandas can be the better choice when the dataset is modest, performance is already acceptable, your team already uses Pandas, another library expects Pandas objects, or ecosystem compatibility matters more than speed.

## 1. Import Polars

Polars may already be installed in Google Colab or another Python environment.

The following cell checks for Polars and installs it only if it is missing.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("polars") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "polars"
    ])

import polars as pl

print("Polars version:", pl.__version__)

## 2. Load the same messy CSV with Polars

We will use the same dataset so the comparison is about syntax and workflow rather than different data.

In [ ]:
from urllib.request import urlretrieve

polars_url = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/"
    "main/IAN601_Module12_Cleaning_EDA_400.csv"
)

polars_file = "IAN601_Module12_Cleaning_EDA_400.csv"

urlretrieve(polars_url, polars_file)

pl_orders = pl.read_csv(polars_file)

print("Shape:", pl_orders.shape)
pl_orders.head()

<details>
<summary><strong>Syntax & parameters — pl.read_csv()</strong></summary>

<pre><code class="language-python">pl.read_csv(
    source,
    separator=",",
    has_header=True,
    columns=None,
    null_values=None,
    try_parse_dates=False
)</code></pre>

<ul>
<li><code>source</code> — source file or supported file-like input.</li>
<li><code>separator</code> — field delimiter.</li>
<li><code>has_header</code> — whether the first row contains column names.</li>
<li><code>columns</code> — selected columns to read.</li>
<li><code>null_values</code> — values interpreted as null.</li>
<li><code>try_parse_dates</code> — attempt date parsing.</li>
<li><code>schema_overrides</code> — override inferred types for selected columns.</li>
</ul>

<p>For a performance-focused lazy workflow, use <code>scan_csv()</code> rather than immediately materializing the full file.</p>
</details>

## 3. Select columns and understand expressions

Pandas often selects columns using brackets.

Polars commonly uses an expression system. For example, `pl.col("amount")` describes a column expression that Polars can combine with other operations and optimize as part of a query.

In [ ]:
pl_orders.select(
    "order_id",
    "region",
    "channel",
    "amount"
).head()

## 4. Clean text and convert types

The cleaning goals are the same as with Pandas:

- standardize text;
- convert amount to numeric;
- convert dates;
- identify invalid values.

The syntax is different, but the analytical reasoning is unchanged.

In [ ]:
pl_clean = (
    pl_orders
    .with_columns(
        pl.col("region")
        .str.strip_chars()
        .str.to_titlecase(),

        pl.col("channel")
        .str.strip_chars()
        .str.to_lowercase(),

        pl.col("status")
        .str.strip_chars()
        .str.to_lowercase(),

        pl.col("amount")
        .str.replace("$", "", literal=True)
        .cast(pl.Float64, strict=False),

        pl.col("order_date")
        .str.to_date(strict=False),

        pl.col("satisfaction")
        .cast(pl.Int64, strict=False)
    )
)

pl_clean.head()

<details>
<summary><strong>Syntax & parameters — with_columns(), pl.col(), and cast()</strong></summary>

<p><strong>General pattern</strong></p>
<pre><code class="language-python">df.with_columns(
    expression_1,
    expression_2
)</code></pre>

<p><code>with_columns()</code> creates or replaces columns.</p>

<pre><code class="language-python">pl.col("amount")
pl.col(["amount", "satisfaction"])

(
    pl.col("amount") * 1.07
).alias("amount_with_tax")</code></pre>

<p><strong>Safe conversion</strong></p>
<pre><code class="language-python">pl.col("amount").cast(
    pl.Float64,
    strict=False
)</code></pre>

<p>With <code>strict=False</code>, values that cannot be converted become null instead of causing the entire conversion to fail.</p>

<p><strong>String expressions</strong></p>
<pre><code class="language-python">pl.col("text").str.strip_chars()
pl.col("text").str.to_lowercase()
pl.col("text").str.to_uppercase()
pl.col("text").str.replace("old", "new")
pl.col("text").str.contains("pattern")</code></pre>
</details>

## 5. Profile missing values

Polars uses **null** as its main missing-value representation.

You can count nulls across every column in one operation.

In [ ]:
pl_clean.null_count()

In [ ]:
pl_clean.filter(
    pl.col("amount").is_null()
).select(
    "order_id",
    "amount"
).head()

<details>
<summary><strong>Syntax & parameters — null handling in Polars</strong></summary>

<pre><code class="language-python">df.null_count()

pl.col("amount").is_null()
pl.col("amount").is_not_null()</code></pre>

<p><strong>Fill nulls</strong></p>
<pre><code class="language-python">df.with_columns(
    pl.col("city").fill_null("Unknown")
)</code></pre>

<p><strong>Drop nulls</strong></p>
<pre><code class="language-python">df.drop_nulls()
df.drop_nulls(["order_id", "amount"])</code></pre>

<p><strong>Fill using a statistic</strong></p>
<pre><code class="language-python">df.with_columns(
    pl.col("amount").fill_null(
        pl.col("amount").median()
    )
)</code></pre>

<p>Polars distinguishes <code>null</code> from floating-point <code>NaN</code>. Use <code>fill_null()</code> and <code>fill_nan()</code> for their respective cases.</p>
</details>

## 6. Filter invalid rows

Polars filtering uses expressions instead of Pandas-style Boolean bracket indexing.

In [ ]:
invalid_scores = pl_clean.filter(
    pl.col("satisfaction").is_not_null()
    & ~pl.col("satisfaction").is_between(1, 5)
)

invalid_scores.select(
    "order_id",
    "satisfaction"
)

## 7. Group and aggregate

The analytical idea is the same as Pandas:

1. define groups;
2. calculate one or more summaries.

Polars uses the method name `group_by()`.

In [ ]:
polars_channel_summary = (
    pl_clean
    .group_by("channel")
    .agg(
        pl.len().alias("orders"),
        pl.col("amount").mean().alias("avg_amount"),
        pl.col("amount").sum().alias("total_amount"),
        pl.col("satisfaction").mean().alias("avg_satisfaction")
    )
    .sort("total_amount", descending=True)
)

polars_channel_summary

<details>
<summary><strong>Syntax & parameters — group_by() and agg()</strong></summary>

<pre><code class="language-python">(
    df
    .group_by("region")
    .agg(
        pl.len().alias("rows"),
        pl.col("amount").sum().alias("sales"),
        pl.col("amount").mean().alias("avg_amount")
    )
)</code></pre>

<p><strong>Useful aggregation expressions</strong></p>
<pre><code class="language-python">pl.len()
pl.col("x").count()
pl.col("x").sum()
pl.col("x").mean()
pl.col("x").median()
pl.col("x").min()
pl.col("x").max()
pl.col("x").std()
pl.col("x").n_unique()</code></pre>
</details>

## 8. Pandas and Polars side by side

### Pandas

```python
summary = (
    clean
    .groupby("channel", as_index=False)
    .agg(
        orders=("order_id", "count"),
        avg_amount=("amount", "mean")
    )
)
```

### Polars

```python
summary = (
    pl_clean
    .group_by("channel")
    .agg(
        pl.len().alias("orders"),
        pl.col("amount").mean().alias("avg_amount")
    )
)
```

The syntax is different, but the analytical sequence is the same:

**DataFrame → group → aggregate → result**

## 9. Why Polars can be faster: lazy execution

Polars supports both:

- **eager execution** — run operations immediately;
- **lazy execution** — describe the full query first and execute later.

With a lazy query, Polars can inspect the complete plan and optimize it before execution.

Examples include:

- **predicate pushdown** — move filters as close to the data source as possible;
- **projection pushdown** — read only columns that are actually needed;
- simplifying expressions;
- optimizing join order in supported workflows.

For larger file-based analytical pipelines, this can reduce unnecessary CPU work, memory use, and I/O.

In [ ]:
lazy_summary = (
    pl.scan_csv(polars_file)
    .with_columns(
        pl.col("amount")
        .str.replace("$", "", literal=True)
        .cast(pl.Float64, strict=False)
    )
    .filter(
        pl.col("amount") >= 100
    )
    .group_by("region")
    .agg(
        pl.len().alias("orders"),
        pl.col("amount").sum().alias("total_amount")
    )
    .sort(
        "total_amount",
        descending=True
    )
)

lazy_summary.collect()

### Inspect the lazy query plan

`explain()` shows the query plan Polars intends to execute. This is useful for learning how a lazy query is optimized.

In [ ]:
print(
    lazy_summary.explain()
)

<details>
<summary><strong>Syntax & parameters — LazyFrame, scan_csv(), collect(), and streaming</strong></summary>

<p><strong>Start a lazy query</strong></p>
<pre><code class="language-python">lf = pl.scan_csv("file.csv")</code></pre>

<p><strong>Build the query</strong></p>
<pre><code class="language-python">query = (
    lf
    .filter(...)
    .select(...)
    .with_columns(...)
    .group_by(...)
    .agg(...)
)</code></pre>

<p>The result is not fully materialized yet.</p>

<p><strong>Execute</strong></p>
<pre><code class="language-python">result = query.collect()</code></pre>

<p><strong>Inspect the optimized plan</strong></p>
<pre><code class="language-python">print(query.explain())</code></pre>

<p><strong>Streaming execution</strong></p>
<pre><code class="language-python">result = query.collect(
    engine="streaming"
)</code></pre>

<p>Streaming can process supported queries in batches and can be useful when data is too large to comfortably materialize all at once.</p>

<p><strong>Important:</strong> benchmark the real workload instead of assuming one DataFrame library is always faster.</p>
</details>

## 10. Why performance can differ

Polars is designed around a columnar query engine.

Several design choices can help on larger analytical workloads:

- execution implemented in Rust;
- parallel use of CPU cores for many operations;
- expression-based query planning;
- lazy query optimization;
- predicate and projection pushdown;
- streaming for supported queries;
- efficient columnar memory handling.

### Performance depends on the workload

A 400-row classroom CSV is too small to demonstrate meaningful performance advantages.

For small data:

- startup overhead may dominate;
- both libraries may feel instantaneous;
- readability and compatibility matter more than milliseconds.

For larger data, repeated joins, group aggregations, filtering, Parquet scans, and long transformation pipelines are more meaningful candidates for comparison.

A careful benchmark should use the same data, same task, same machine, and repeated timings.

## 11. Moving between Pandas and Polars

You do not have to choose one library permanently.

Conversion can be useful when another library expects a particular DataFrame type.

In [ ]:
polars_to_pandas = (
    polars_channel_summary
    .to_pandas()
)

polars_to_pandas.head()

In [ ]:
pandas_to_polars = pl.from_pandas(
    clean
)

pandas_to_polars.head()

<details>
<summary><strong>Syntax & notes — converting between Pandas and Polars</strong></summary>

<pre><code class="language-python">pl_df = pl.from_pandas(pd_df)
pd_df = pl_df.to_pandas()</code></pre>

<p>Repeatedly converting very large tables can add overhead. If performance is the reason for using Polars, keep the heavy transformation steps in Polars as much as practical.</p>
</details>

## Optional Polars checkpoint

After this section, you should be able to explain:

- what Polars is;
- why it may be faster for many larger analytical workloads;
- why saying “Polars is always faster” would be too strong;
- the role of `pl.col()` expressions;
- how `group_by()` compares with Pandas `groupby()`;
- the difference between eager and lazy execution;
- what `scan_csv()` and `collect()` do;
- what predicate and projection pushdown mean at a high level;
- why Pandas may still be the better choice for some teams and projects.

## Practice challenge

Try these on `pipeline`:

1. Find the three channels with the highest total amount.
2. Compare average satisfaction by region.
3. Count orders by status.
4. Find the five largest orders.
5. Decide whether you would remove the amount outliers. Explain why.

### Optional Polars practice

6. Load the same CSV with Polars.
7. Standardize the `channel` column with Polars string expressions.
8. Convert `amount` to `Float64` using `strict=False`.
9. Filter orders with `amount >= 100`.
10. Group by `region` and calculate count, mean amount, and total amount.
11. Rewrite one eager Polars workflow with `scan_csv()` and `collect()`.
12. Use `explain()` to inspect the lazy query plan.
13. In two or three sentences, explain when you would choose Pandas and when you might consider Polars.

<details>
<summary><strong>Hints & syntax — practice challenge</strong></summary>

<p>Useful methods:</p>
<pre><code class="language-python">value_counts()
groupby().agg()
sort_values()
nlargest()
isna().sum()
duplicated().sum()</code></pre>

<p>For the outlier question, calculate the IQR bounds first, inspect the flagged rows, and then write a reasoned decision. Do not automatically delete them.</p>
</details>

In [ ]:
# 1. Total amount by channel


# 2. Average satisfaction by region


# 3. Orders by status


# 4. Five largest orders


# 5. Write your decision about outliers as a comment


## Summary and self-check

Cleaning is not the same as deleting inconvenient rows. Every transformation should have a reason that can be explained.

Check that you can answer:

- What should be inspected before changing the data?
- Why is missingness measured by both count and percentage?
- Why can `errors="coerce"` be useful during conversion?
- Why should text categories be standardized?
- Why is an outlier not automatically an error?
- What does the IQR rule flag?
- Why should the cleaned dataset be validated again at the end?

A trustworthy analysis begins with a trustworthy understanding of the data.

### Optional Polars self-check

You should also be able to explain at a high level:

- how the same cleaning reasoning applies in both Pandas and Polars;
- why Polars uses expression-oriented syntax;
- how lazy execution can enable query optimization;
- why multi-core execution, pushdown optimizations, and streaming can matter on larger data;
- why actual performance should be measured for the real workload instead of assumed.